In [29]:
from tensorflow.keras.models import Model,load_model
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from sklearn.metrics import classification_report,confusion_matrix
import clahe

In [30]:
train_path = "../Datasets/data/train/Medical/X-Ray/hand/Train"
test_path = "../Datasets/data/test/Medical/X-Ray/hand/Test"
valid_path = "../Datasets/data/vaild/Medical/X-Ray/hand/valid"

In [31]:
train_aug = ImageDataGenerator(
    preprocessing_function = lambda img : clahe.clahe_preprocess(img,preprocess_input),
    rotation_range = 10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    shear_range=0.05,
)
valid_aug = ImageDataGenerator(
    preprocessing_function=lambda img: clahe.clahe_preprocess(img, preprocess_input),
)
test_aug = ImageDataGenerator(preprocessing_function = lambda img : clahe.clahe_preprocess(img,preprocess_input))

In [32]:
train_set = train_aug.flow_from_directory(
    train_path,
    target_size = (224,224),
    batch_size=13,
    class_mode="binary",
)
validation_set = train_aug.flow_from_directory(
    valid_path,
    target_size=(224, 224),
    batch_size=13,
    class_mode="binary",
)
test_set = test_aug.flow_from_directory(
    test_path,
    target_size = (224,224),
    batch_size=13,
    class_mode="binary",
    shuffle = False
)

Found 6612 images belonging to 2 classes.
Found 577 images belonging to 2 classes.
Found 879 images belonging to 2 classes.


In [33]:
base_model = DenseNet201(include_top=False,weights="imagenet",input_shape=(224,224,3),pooling='avg')

In [34]:
base_model.trainable = False

In [35]:
x = base_model.output
x = Dense(64,activation='relu')(x)
output = Dense(1,activation='sigmoid')(x)

In [36]:
model = Model(inputs = base_model.input,outputs = output)

In [37]:
model.compile(optimizer= Adam(1e-4),loss="binary_crossentropy",metrics=['accuracy'])

In [38]:
erly_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [94]:
model.fit(train_set,validation_data=validation_set,epochs=30,callbacks=erly_stop)

Epoch 1/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 76s 147ms/step - accuracy: 0.8621 - loss: 0.3296 - val_accuracy: 0.8007 - val_loss: 0.4319
Epoch 2/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 69s 135ms/step - accuracy: 0.8604 - loss: 0.3308 - val_accuracy: 0.7799 - val_loss: 0.4486
Epoch 3/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 69s 136ms/step - accuracy: 0.8562 - loss: 0.3330 - val_accuracy: 0.7695 - val_loss: 0.4425
Epoch 4/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 81s 158ms/step - accuracy: 0.8560 - loss: 0.3381 - val_accuracy: 0.8059 - val_loss: 0.4051
Epoch 5/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 76s 150ms/step - accuracy: 0.8637 - loss: 0.3269 - val_accuracy: 0.7920 - val_loss: 0.4197
Epoch 6/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 78s 152ms/step - accuracy: 0.8548 - loss: 0.3420 - val_accuracy: 0.8042 - val_loss: 0.4230
Epoch 7/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 83s 164ms/step - accuracy: 0.8575 - loss: 0.3321 - val_accuracy: 0.7816 - val_loss: 0.4263
Epoch 8/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 73s 144ms/step - accuracy: 0.8515 - loss: 0

In [95]:
prob = model.predict(test_set)

68/68 ━━━━━━━━━━━━━━━━━━━━ 5s 75ms/step


In [100]:
pred = (prob >= 0.5).astype(int)

In [43]:
true_data = test_set.classes
model1 = model

In [101]:
print(classification_report(true_data,pred))
print(confusion_matrix(true_data,pred))

              precision    recall  f1-score   support

           0       0.91      0.68      0.77       508
           1       0.67      0.90      0.77       371

    accuracy                           0.77       879
   macro avg       0.79      0.79      0.77       879
weighted avg       0.81      0.77      0.77       879

[[343 165]
 [ 36 335]]


In [45]:
base_model.trainable = True

In [46]:
for layer in base_model.layers[:-30]:
    layer.trainable = False

In [47]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2
)

In [48]:
model.compile(optimizer=Adam(1e-5),loss="binary_crossentropy",metrics=['accuracy'])

In [49]:
model.fit(train_set,validation_data=validation_set,epochs=30,callbacks=[early_stop,reduce_lr])

Epoch 1/30


I0000 00:00:1793082943.693682   18326 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_335269__.464


 99/509 ━━━━━━━━━━━━━━━━━━━━ 1:01 150ms/step - accuracy: 0.7234 - loss: 0.5403

I0000 00:00:1793082988.950538   18326 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_335269__.464


509/509 ━━━━━━━━━━━━━━━━━━━━ 211s 287ms/step - accuracy: 0.7331 - loss: 0.5237 - val_accuracy: 0.7036 - val_loss: 0.5571 - learning_rate: 1.0000e-05
Epoch 2/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 86s 169ms/step - accuracy: 0.7403 - loss: 0.5127 - val_accuracy: 0.7175 - val_loss: 0.5375 - learning_rate: 1.0000e-05
Epoch 3/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 72s 141ms/step - accuracy: 0.7619 - loss: 0.4907 - val_accuracy: 0.6898 - val_loss: 0.5332 - learning_rate: 1.0000e-05
Epoch 4/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 68s 133ms/step - accuracy: 0.7615 - loss: 0.4905 - val_accuracy: 0.7244 - val_loss: 0.5262 - learning_rate: 1.0000e-05
Epoch 5/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 79s 155ms/step - accuracy: 0.7672 - loss: 0.4808 - val_accuracy: 0.7123 - val_loss: 0.5406 - learning_rate: 1.0000e-05
Epoch 6/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 75s 148ms/step - accuracy: 0.7756 - loss: 0.4712 - val_accuracy: 0.7227 - val_loss: 0.5242 - learning_rate: 1.0000e-05
Epoch 7/30
509/509 ━━━━━━━━━━━━━━━━━━━━ 67s 132ms/step -

In [67]:
prob = model.predict(test_set)

68/68 ━━━━━━━━━━━━━━━━━━━━ 6s 76ms/step


In [91]:
pred = (prob >= 0.5).astype(int)

In [75]:
true_data = test_set.classes

In [92]:
print(classification_report(true_data,pred))
print(confusion_matrix(true_data,pred))

              precision    recall  f1-score   support

           0       0.90      0.69      0.78       508
           1       0.68      0.89      0.77       371

    accuracy                           0.77       879
   macro avg       0.79      0.79      0.77       879
weighted avg       0.81      0.77      0.78       879

[[349 159]
 [ 39 332]]


In [ ]:
model1.save('hand_model.keras')